In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path("../..").resolve()))

import pandas as pd
import numpy as np
from loguru import logger

from world_cup_2026.config import RAW_DATA_DIR, RANDOM_SEED
from world_cup_2026.data_ingestion.normalize import normalize_dataframe_teams
from world_cup_2026.features.elo import calculate_elo
from world_cup_2026.features.form import FormCalculator
from world_cup_2026.features.h2h import H2HAnalyzer

np.random.seed(RANDOM_SEED)

df_results = pd.read_csv(
    RAW_DATA_DIR / "martj42_results" / "results.csv",
    parse_dates=["date"]
)
df_results_norm = normalize_dataframe_teams(df_results, ["home_team", "away_team"])

print(f"Matches loaded: {len(df_results_norm):,}")
print(f"Date range: {df_results_norm['date'].min().date()} → {df_results_norm['date'].max().date()}")
print("Setup complete.")

2026-04-03 15:16:26.718 | INFO     | world_cup_2026.config:<module>:12 - PROJ_ROOT path is: C:\Users\feder\Documents\data_repos\world-cup-2026-predictor
2026-04-03 15:16:27.676 | WARNING  | world_cup_2026.data_ingestion.normalize:normalize_team_name:193 - Unknown team name: 'Guernsey' — add to normalize.py if needed.
2026-04-03 15:16:27.676 | WARNING  | world_cup_2026.data_ingestion.normalize:normalize_team_name:193 - Unknown team name: 'Jersey' — add to normalize.py if needed.
2026-04-03 15:16:27.683 | WARNING  | world_cup_2026.data_ingestion.normalize:normalize_team_name:193 - Unknown team name: 'Alderney' — add to normalize.py if needed.
2026-04-03 15:16:27.683 | WARNING  | world_cup_2026.data_ingestion.normalize:normalize_team_name:193 - Unknown team name: 'Catalonia' — add to normalize.py if needed.
2026-04-03 15:16:27.683 | WARNING  | world_cup_2026.data_ingestion.normalize:normalize_team_name:193 - Unknown team name: 'Basque Country' — add to normalize.py if needed.
2026-04-03 1

Matches loaded: 49,215
Date range: 1872-11-30 → 2026-03-31
Setup complete.


In [3]:
import time

# --- Elo ---
t0 = time.time()
df_elo = calculate_elo(df_results_norm)
print(f"Elo: {len(df_elo):,} rows, {time.time()-t0:.1f}s")

# --- Form ---
t0 = time.time()
calc = FormCalculator(df_results_norm, windows=[5, 10, 20])
df_form = calc.compute_form_features(df_results_norm)
print(f"Form: {len(df_form):,} rows, {len(df_form.columns)} cols, {time.time()-t0:.1f}s")

# Sanity check
assert len(df_elo) == len(df_form) == len(df_results_norm), "Row count mismatch!"
print("Row count check passed.")

2026-04-03 15:17:36.393 | INFO     | world_cup_2026.features.elo:calculate_elo:196 - Calculating Elo for 49,215 matches...


2026-04-03 15:17:37.299 | SUCCESS  | world_cup_2026.features.elo:calculate_elo:243 - Elo calculation complete. Registry has 333 teams.
2026-04-03 15:17:37.393 | INFO     | world_cup_2026.features.form:__init__:95 - FormCalculator initialized — 49,215 matches, windows=[5, 10, 20], decay_lambda=0.1
2026-04-03 15:17:37.478 | INFO     | world_cup_2026.features.form:compute_form_features:309 - Computing form features (all matches) for 49,215 rows...


Elo: 49,215 rows, 1.0s


2026-04-03 15:17:37.719 | DEBUG    | world_cup_2026.features.form:_build_long_df:155 - Long format built: 98,430 rows, 333 teams.
2026-04-03 15:17:37.719 | DEBUG    | world_cup_2026.features.form:_compute_rolling_features:182 -   Computing rolling window=5...
2026-04-03 15:17:38.691 | DEBUG    | world_cup_2026.features.form:_compute_rolling_features:182 -   Computing rolling window=10...
2026-04-03 15:17:39.672 | DEBUG    | world_cup_2026.features.form:_compute_rolling_features:182 -   Computing rolling window=20...
2026-04-03 15:17:41.841 | SUCCESS  | world_cup_2026.features.form:compute_form_features:358 - Form features computed. Added 66 columns.


Form: 49,215 rows, 75 cols, 4.6s
Row count check passed.


In [4]:
# --- H2H ---
t0 = time.time()
analyzer = H2HAnalyzer(df_elo)
print(f"H2H analyzer initialized, {time.time()-t0:.1f}s")

# --- Target variable ---
# From home team perspective: 2=home win, 1=draw, 0=away win
def encode_result(row):
    if row["home_score"] > row["away_score"]:
        return 2
    elif row["home_score"] == row["away_score"]:
        return 1
    else:
        return 0

df_results_norm["target"] = df_results_norm.apply(encode_result, axis=1)

print(f"\nTarget distribution:")
print(df_results_norm["target"].value_counts().sort_index().rename({0: "Away win", 1: "Draw", 2: "Home win"}))
print(f"\nHome win rate: {(df_results_norm['target']==2).mean():.3f}")
print(f"Draw rate:     {(df_results_norm['target']==1).mean():.3f}")
print(f"Away win rate: {(df_results_norm['target']==0).mean():.3f}")

2026-04-03 15:22:33.185 | INFO     | world_cup_2026.features.h2h:__init__:70 - H2HAnalyzer initialized — 49,215 matches, window=48m, decay_lambda=0.02


H2H analyzer initialized, 0.1s

Target distribution:
target
Away win    13912
Draw        11197
Home win    24106
Name: count, dtype: int64

Home win rate: 0.490
Draw rate:     0.228
Away win rate: 0.283


In [7]:
# --- Filter to modern era for feature matrix ---
CUTOFF_DATE = pd.Timestamp("2016-01-01")

df_master = df_elo[ELO_COLS].copy()
df_master["target"] = df_results_norm["target"].values
df_master = pd.concat([df_master, df_form[FORM_COLS].reset_index(drop=True)], axis=1)

# Filter AFTER joining — Elo and Form were computed on full history (correct)
# We only exclude pre-2000 from the training matrix
df_master = df_master[df_master["date"] >= CUTOFF_DATE].reset_index(drop=True)

print(f"Matches from 2000 onwards: {len(df_master):,}")
print(f"Date range: {df_master['date'].min().date()} → {df_master['date'].max().date()}")
print(f"Shape after filter: {df_master.shape}")

# --- H2H features ---
print("\nComputing H2H features...")
t0 = time.time()

h2h_records = []
for idx, row in df_master.iterrows():
    features = analyzer.get_matchup_features(
        team_a=row["home_team"],
        team_b=row["away_team"],
        as_of=row["date"]
    )
    h2h_records.append(features)

    if idx % 5000 == 0 and idx > 0:
        elapsed = time.time() - t0
        pct = idx / len(df_master) * 100
        print(f"  {idx:,}/{len(df_master):,} ({pct:.0f}%) — {elapsed:.1f}s elapsed")

df_h2h = pd.DataFrame(h2h_records)
df_master = pd.concat([df_master, df_h2h.reset_index(drop=True)], axis=1)

elapsed = time.time() - t0
print(f"\nH2H done: {elapsed:.1f}s")
print(f"Final shape: {df_master.shape}")
print(f"H2H columns: {list(df_h2h.columns)}")

Matches from 2000 onwards: 9,796
Date range: 2016-01-03 → 2026-03-31
Shape after filter: (9796, 78)

Computing H2H features...


2026-04-03 15:59:48.806 | DEBUG    | world_cup_2026.features.h2h:get_h2h_features:192 - H2H India vs Afghanistan: only 0 matches (min=2) — returning neutral features.
2026-04-03 15:59:49.102 | DEBUG    | world_cup_2026.features.h2h:get_h2h_features:192 - H2H Rwanda vs Cameroon: only 0 matches (min=2) — returning neutral features.
2026-04-03 15:59:49.225 | DEBUG    | world_cup_2026.features.h2h:get_h2h_features:192 - H2H Sweden vs Estonia: only 1 matches (min=2) — returning neutral features.
2026-04-03 15:59:50.184 | DEBUG    | world_cup_2026.features.h2h:get_h2h_features:192 - H2H Rwanda vs DR Congo: only 0 matches (min=2) — returning neutral features.
2026-04-03 15:59:50.468 | DEBUG    | world_cup_2026.features.h2h:get_h2h_features:192 - H2H Maldives vs Cambodia: only 0 matches (min=2) — returning neutral features.
2026-04-03 15:59:50.617 | DEBUG    | world_cup_2026.features.h2h:get_h2h_features:192 - H2H Uganda vs Cameroon: only 0 matches (min=2) — returning neutral features.
2026-04

  5,000/9,796 (51%) — 741.4s elapsed


2026-04-03 16:12:11.054 | DEBUG    | world_cup_2026.features.h2h:get_h2h_features:192 - H2H Egypt vs Liberia: only 1 matches (min=2) — returning neutral features.
2026-04-03 16:12:11.388 | DEBUG    | world_cup_2026.features.h2h:get_h2h_features:192 - H2H Sri Lanka vs Bangladesh: only 1 matches (min=2) — returning neutral features.
2026-04-03 16:12:11.436 | DEBUG    | world_cup_2026.features.h2h:get_transitive_features:303 - Transitive Sri Lanka vs Bangladesh: only 2 common rivals (min=3) — returning neutral.
2026-04-03 16:12:11.462 | DEBUG    | world_cup_2026.features.h2h:get_h2h_features:192 - H2H Maldives vs Nepal: only 1 matches (min=2) — returning neutral features.
2026-04-03 16:12:11.576 | DEBUG    | world_cup_2026.features.h2h:get_h2h_features:192 - H2H Sri Lanka vs Nepal: only 0 matches (min=2) — returning neutral features.
2026-04-03 16:12:11.818 | DEBUG    | world_cup_2026.features.h2h:get_h2h_features:192 - H2H Sudan vs Guinea: only 0 matches (min=2) — returning neutral featu


H2H done: 2035.4s
Final shape: (9796, 92)
H2H columns: ['h2h_matches', 'h2h_win_rate_a', 'h2h_goal_diff_a', 'h2h_elo_edge_a', 'h2h_weighted_edge_a', 'h2h_decay_weight', 'h2h_reliable', 'transitive_common_rivals', 'transitive_edge_a', 'transitive_goal_diff_edge', 'transitive_reliable', 'team_a', 'team_b', 'as_of']


In [8]:
# Drop H2H metadata columns that aren't features
H2H_META_COLS = ["team_a", "team_b", "as_of"]
df_master = df_master.drop(columns=H2H_META_COLS, errors="ignore")

print(f"Shape after cleanup: {df_master.shape}")

# Check NaN counts
nan_counts = df_master.isnull().sum()
nan_cols = nan_counts[nan_counts > 0]
if len(nan_cols) == 0:
    print("No NaN values found.")
else:
    print(f"\nColumns with NaN ({len(nan_cols)} total):")
    print(nan_cols.sort_values(ascending=False).head(20))

# Check target distribution in filtered dataset
print(f"\nTarget distribution (2016+):")
print(df_master["target"].value_counts().sort_index().rename({0: "Away win", 1: "Draw", 2: "Home win"}))
print(f"\nHome win rate: {(df_master['target']==2).mean():.3f}")
print(f"Draw rate:     {(df_master['target']==1).mean():.3f}")
print(f"Away win rate: {(df_master['target']==0).mean():.3f}")

# Preview
print(f"\nColumn groups:")
print(f"  Identity:  {[c for c in df_master.columns if c in ['date','home_team','away_team','tournament','neutral']]}")
print(f"  Elo:       {[c for c in df_master.columns if 'elo' in c or 'win_prob' in c]}")
print(f"  Form home: {len([c for c in df_master.columns if c.startswith('home_form_')])} cols")
print(f"  Form away: {len([c for c in df_master.columns if c.startswith('away_form_')])} cols")
print(f"  H2H:       {[c for c in df_master.columns if c.startswith('h2h_') or c.startswith('transitive_')]}")
print(f"  Target:    {['target']}")

Shape after cleanup: (9796, 89)
No NaN values found.

Target distribution (2016+):
target
Away win    2845
Draw        2285
Home win    4666
Name: count, dtype: int64

Home win rate: 0.476
Draw rate:     0.233
Away win rate: 0.290

Column groups:
  Identity:  ['date', 'home_team', 'away_team', 'tournament', 'neutral']
  Elo:       ['elo_pre_home', 'elo_pre_away', 'elo_diff', 'win_prob_home', 'h2h_elo_edge_a']
  Form home: 33 cols
  Form away: 33 cols
  H2H:       ['h2h_matches', 'h2h_win_rate_a', 'h2h_goal_diff_a', 'h2h_elo_edge_a', 'h2h_weighted_edge_a', 'h2h_decay_weight', 'h2h_reliable', 'transitive_common_rivals', 'transitive_edge_a', 'transitive_goal_diff_edge', 'transitive_reliable']
  Target:    ['target']


In [9]:
from world_cup_2026.config import PROCESSED_DATA_DIR

# Create directory if it doesn't exist
PROCESSED_DATA_DIR.mkdir(parents=True, exist_ok=True)

output_path = PROCESSED_DATA_DIR / "master_features.parquet"
df_master.to_parquet(output_path, index=False)

print(f"Saved: {output_path}")
print(f"File size: {output_path.stat().st_size / 1024:.1f} KB")

# Verify roundtrip
df_check = pd.read_parquet(output_path)
assert df_check.shape == df_master.shape, "Roundtrip shape mismatch!"
assert list(df_check.columns) == list(df_master.columns), "Roundtrip columns mismatch!"
print(f"Roundtrip verification passed.")
print(f"\nFinal feature matrix: {df_master.shape[0]:,} rows x {df_master.shape[1]} columns")
print(f"Ready for modeling.")

Saved: C:\Users\feder\Documents\data_repos\world-cup-2026-predictor\data\processed\master_features.parquet
File size: 1577.7 KB
Roundtrip verification passed.

Final feature matrix: 9,796 rows x 89 columns
Ready for modeling.
